# 🚀 Huấn Luyện & Đánh Giá Row 7 (SG-GBA) Trên COCOA & KINS (NVIDIA A100 GPU)

**Dự án:** Khóa luận Tốt nghiệp — Dự đoán biên dạng hoàn chỉnh của vật thể bị che khuất (**Amodal Shape Prediction**)  
**Người thực hiện:** Võ Thế Nguyễn (Ryan) & Phan Quang Nghĩa  
**Môi trường:** Google Colab Pro — **GPU NVIDIA A100 (40GB/80GB VRAM)**  

---

### 📌 Mục Tiêu & Cơ Chế Hoạt Động Của Notebook Này:
1. **Chỉ chạy duy nhất Row 7 (Kiến trúc đề xuất mới của KLTN: SG-GBA Bottleneck):**
   - **Tập 1 — COCOA (91 lớp):** Chạy duy nhất Row 7 (`EXPECTED_PARAMS = 34,867,566`), thời gian ~25 phút.
   - **Tập 2 — KINS (20 lớp):** Chạy duy nhất Row 7 (`EXPECTED_PARAMS = 34,813,038`), thời gian ~1.5 giờ.
   - *(Tập D2SA đã được tách riêng chạy ở notebook `train_d2sa_colab_a100.ipynb`).*
2. **Cơ chế nạp dữ liệu thông minh (Zero-Interruption Data Engine):**
   - **Ưu tiên 1:** Tự động quét toàn bộ danh sách `SEARCH_DIRS` trên Google Drive (bao gồm các thư mục `data/COCOA/annotations.zip`, `data/KINS`,...). Nếu đã có dữ liệu -> sử dụng ngay lập tức, không tải lại.
   - **Ưu tiên 2 (Tự động tải về):** Nếu chưa có trên Drive -> hệ thống tự động tải trực tiếp từ máy chủ chính thức (ảnh COCO từ `images.cocodataset.org`, dữ liệu KINS từ `kagglehub`) thẳng vào SSD NVMe của Colab với tốc độ cao nhất.
3. **Bảo toàn kết quả tuyệt đối (Fault-Tolerant Persistence):**
   - Toàn bộ checkpoints (`.pth`), logs (`.log`, `.jsonl`), và kết quả đánh giá (`.json`) được symlink và ghi trực tiếp vào Google Drive (`/content/drive/MyDrive/KLTN_Amodal_2026/`).

### ⚡ Bước 1: Kiểm Tra GPU A100 & Thông Số Phần Cứng Colab
Đảm bảo bạn đã chọn Runtime: **Runtime -> Change runtime type -> A100 GPU -> High-RAM**.

In [ ]:
!nvidia-smi

import os
import psutil
import torch

print("\n" + "=" * 65)
print("THÔNG TIN HỆ THỐNG & PHẦN CỨNG COLAB")
print("=" * 65)
print("PyTorch Version :", torch.__version__)
print("CUDA Available  :", torch.cuda.is_available())

if torch.cuda.is_available():
    gpu_name = torch.cuda.get_device_name(0)
    vram_gb = torch.cuda.get_device_properties(0).total_memory / (1024**3)
    compute_cap = torch.cuda.get_device_capability(0)
    print(f"GPU Name        : {gpu_name}")
    print(f"Total VRAM      : {vram_gb:.2f} GB")
    print(f"Compute Cap     : {compute_cap[0]}.{compute_cap[1]}")

    # Kích hoạt tăng tốc phần cứng cho NVIDIA Ampere A100
    torch.backends.cudnn.benchmark = True
    if hasattr(torch, "set_float32_matmul_precision"):
        torch.set_float32_matmul_precision("high")
        print("⚡ Đã kích hoạt Tensor Cores TF32 (float32_matmul_precision = 'high')")
else:
    print("⚠️ CẢNH BÁO: Chưa bật GPU! Vui lòng vào Runtime -> Change runtime type -> A100 GPU")

cpu_count = os.cpu_count()
ram_gb = psutil.virtual_memory().total / (1024**3)
print(f"vCPUs Available : {cpu_count} cores (Tối ưu cho num_workers=8)")
print(f"System RAM      : {ram_gb:.2f} GB")
print("=" * 65)

### 📁 Bước 2: Kết Nối Google Drive, Clone Repo & Thiết Lập Symlink Lưu Trữ Bền Vững
Mọi checkpoint mô hình, telemetry log và kết quả đánh giá khoa học sẽ được lưu liên tục sang Google Drive.

In [ ]:
from google.colab import drive
import os
import shutil

# 1. Mount Google Drive
drive.mount('/content/drive')

# 2. Thiết lập thư mục lưu trữ KLTN trên Google Drive của bạn
DRIVE_DIR = "/content/drive/MyDrive/KLTN_Amodal_2026"
if not os.path.exists(DRIVE_DIR) and os.path.exists("/content/drive/MyDrive/amodal_shape_prediction"):
    DRIVE_DIR = "/content/drive/MyDrive/amodal_shape_prediction"

os.makedirs(f"{DRIVE_DIR}/checkpoints/cocoa/row7_sg_gba_bottleneck", exist_ok=True)
os.makedirs(f"{DRIVE_DIR}/checkpoints/kins/row7_sg_gba", exist_ok=True)
os.makedirs(f"{DRIVE_DIR}/logs", exist_ok=True)
os.makedirs(f"{DRIVE_DIR}/results", exist_ok=True)

# 3. Clone hoặc pull source code mới nhất từ GitHub
%cd /content
if not os.path.exists("/content/amodal_shape_prediction"):
    !git clone https://github.com/vothenguyen/amodal_shape_prediction.git
else:
    %cd /content/amodal_shape_prediction
    !git pull origin main

%cd /content/amodal_shape_prediction

# 4. Tạo symbolic link bền vững sang Google Drive
!rm -rf checkpoints logs results
!ln -s "{DRIVE_DIR}/checkpoints" checkpoints
!ln -s "{DRIVE_DIR}/logs" logs
!ln -s "{DRIVE_DIR}/results" results

# 5. Cài đặt các thư viện bổ trợ
!pip install -q timm albumentations pycocotools tqdm gradio ultralytics kagglehub
print("\n✅ Môi trường A100 và hệ thống lưu trữ bền vững Drive đã sẵn sàng 100%!")

---
# 🥥 PHẦN 1: TẬP DỮ LIỆU COCOA (COCO AMODAL - 91 LỚP)

### 🔍 Bước 3: Tự Động Quét Drive Hoặc Tải Dữ Liệu COCOA (Smart Search & Fallback)
Cơ chế tự động:
1. **Annotations (`annotations.zip`):** Tự động quét toàn bộ danh sách `SEARCH_DIRS_COCOA` (bao gồm thư mục `data/COCOA` trên Drive mà bạn vừa upload). Nếu tìm thấy file nén hoặc file JSON -> ưu tiên giải nén/sử dụng ngay.
2. **Images (`train2014`, `val2014`):** Quét trên Drive trước, nếu chưa có thì tải trực tiếp siêu tốc từ `http://images.cocodataset.org/zips/...` thẳng vào SSD Colab.

In [ ]:
import os
import sys
import glob
import shutil
import time

LOCAL_DATA_DIR = "/content/amodal_shape_prediction/data"
os.makedirs(LOCAL_DATA_DIR, exist_ok=True)
ANN_DIR = os.path.join(LOCAL_DATA_DIR, "annotations")
os.makedirs(ANN_DIR, exist_ok=True)

REQ_COCO_JSONS = [
    "COCO_amodal_train2014.json",
    "COCO_amodal_val2014.json"
]

# Danh sách các thư mục ưu tiên tìm kiếm dữ liệu COCOA trên Drive và cục bộ
SEARCH_DIRS_COCOA = [
    f"{DRIVE_DIR}/data/COCOA",
    f"{DRIVE_DIR}/data/cocoa",
    f"{DRIVE_DIR}/data",
    "/content/drive/MyDrive/data/COCOA",
    "/content/drive/MyDrive/data/cocoa",
    "/content/drive/MyDrive/COCOA",
    "/content/drive/MyDrive/cocoa",
    "/content/drive/MyDrive/CV+/data/COCOA",
    "/content/drive/MyDrive/CV+/data",
    "/content/drive/MyDrive/amodal_shape_prediction/data/COCOA",
    "/content/drive/MyDrive/amodal_shape_prediction/data",
    f"{DRIVE_DIR}",
    "/content/drive/MyDrive/data",
    "/content/drive/MyDrive",
    "/content/amodal_shape_prediction/data/COCOA",
    "/content/amodal_shape_prediction/data",
    "/content/data/COCOA",
    "/content/data",
    "/content"
]

def find_file_in_search_dirs(patterns, search_dirs):
    for sdir in search_dirs:
        if not os.path.exists(sdir):
            continue
        for p in patterns:
            target = os.path.join(sdir, p)
            if os.path.isfile(target) or os.path.isdir(target):
                return target
            matches = glob.glob(os.path.join(sdir, p))
            if matches and (os.path.isfile(matches[0]) or os.path.isdir(matches[0])):
                return matches[0]
    return None

print("=" * 70)
print("🔍 BẮT ĐẦU TỰ ĐỘNG THIẾT LẬP DỮ LIỆU COCOA")
print("=" * 70)

# -------------------------------------------------------------------------
# 1. XỬ LÝ ANNOTATIONS COCOA
# -------------------------------------------------------------------------
has_train_json = os.path.isfile(os.path.join(ANN_DIR, "COCO_amodal_train2014.json"))
has_val_json = os.path.isfile(os.path.join(ANN_DIR, "COCO_amodal_val2014.json"))

if has_train_json and has_val_json:
    print("✅ File annotations COCOA đã sẵn sàng tại data/annotations/")
else:
    print("🔎 Đang quét tìm annotations.zip trên Google Drive...")
    ann_archive = find_file_in_search_dirs([
        "annotations.zip",
        "*cocoa*annotation*.zip",
        "*annotations*.zip",
        "annotations.tar.gz"
    ], SEARCH_DIRS_COCOA)

    if ann_archive:
        size_mb = os.path.getsize(ann_archive) / (1024 * 1024)
        print(f"📦 Tìm thấy file nén annotations ({size_mb:.1f} MB) tại:\n   {ann_archive}")
        print("   Đang giải nén vào data/...")
        t0 = time.time()
        if ann_archive.endswith(".zip"):
            !unzip -qo "{ann_archive}" -d "{LOCAL_DATA_DIR}"
        else:
            !tar -xzf "{ann_archive}" -C "{LOCAL_DATA_DIR}"
        print(f"   ✅ Giải nén xong trong {time.time() - t0:.1f}s!")
    else:
        # Thử tìm file JSON rời
        f_train = find_file_in_search_dirs(["COCO_amodal_train2014.json", "*amodal_train2014.json"], SEARCH_DIRS_COCOA)
        f_val = find_file_in_search_dirs(["COCO_amodal_val2014.json", "*amodal_val2014.json"], SEARCH_DIRS_COCOA)
        if f_train and f_val:
            shutil.copy(f_train, os.path.join(ANN_DIR, "COCO_amodal_train2014.json"))
            shutil.copy(f_val, os.path.join(ANN_DIR, "COCO_amodal_val2014.json"))
            print(f"✅ Đã sao chép các file JSON từ: {os.path.dirname(f_train)}")
        else:
            print("⚠️ Chưa tìm thấy annotations.zip trên Drive. Vui lòng tải file lên Drive.")

    # Chuẩn hóa nếu file giải nén bị lồng trong thư mục con
    for root, dirs, files in os.walk(LOCAL_DATA_DIR):
        for req_f in REQ_COCO_JSONS:
            if req_f in files:
                target_path = os.path.join(ANN_DIR, req_f)
                src_path = os.path.join(root, req_f)
                if src_path != target_path and not os.path.exists(target_path):
                    shutil.copy(src_path, target_path)

# -------------------------------------------------------------------------
# 2. XỬ LÝ ẢNH COCO2014 (train2014 & val2014)
# -------------------------------------------------------------------------
train_img_dir = os.path.join(LOCAL_DATA_DIR, "train2014")
val_img_dir = os.path.join(LOCAL_DATA_DIR, "val2014")

has_train_imgs = os.path.isdir(train_img_dir) and len(os.listdir(train_img_dir)) >= 10000
has_val_imgs = os.path.isdir(val_img_dir) and len(os.listdir(val_img_dir)) >= 5000

if not has_train_imgs:
    # Quét Drive trước
    train_drive = find_file_in_search_dirs(["train2014", "train2014.zip"], SEARCH_DIRS_COCOA)
    if train_drive and os.path.isdir(train_drive):
        print(f"🔗 Tìm thấy thư mục train2014 trên Drive: {train_drive}")
        !rm -rf "{train_img_dir}" && ln -sf "{train_drive}" "{train_img_dir}"
    elif train_drive and train_drive.endswith(".zip"):
        print(f"📦 Giải nén train2014.zip từ Drive: {train_drive}")
        !unzip -q "{train_drive}" -d "{LOCAL_DATA_DIR}"
    else:
        print("🌐 Tự động tải ảnh train2014 từ images.cocodataset.org thẳng vào SSD Colab...")
        %cd "{LOCAL_DATA_DIR}"
        !wget -q http://images.cocodataset.org/zips/train2014.zip && unzip -q train2014.zip && rm train2014.zip
        %cd /content/amodal_shape_prediction

if not has_val_imgs:
    val_drive = find_file_in_search_dirs(["val2014", "val2014.zip"], SEARCH_DIRS_COCOA)
    if val_drive and os.path.isdir(val_drive):
        print(f"🔗 Tìm thấy thư mục val2014 trên Drive: {val_drive}")
        !rm -rf "{val_img_dir}" && ln -sf "{val_drive}" "{val_img_dir}"
    elif val_drive and val_drive.endswith(".zip"):
        print(f"📦 Giải nén val2014.zip từ Drive: {val_drive}")
        !unzip -q "{val_drive}" -d "{LOCAL_DATA_DIR}"
    else:
        print("🌐 Tự động tải ảnh val2014 từ images.cocodataset.org thẳng vào SSD Colab...")
        %cd "{LOCAL_DATA_DIR}"
        !wget -q http://images.cocodataset.org/zips/val2014.zip && unzip -q val2014.zip && rm val2014.zip
        %cd /content/amodal_shape_prediction

# -------------------------------------------------------------------------
# 3. TẠO SYMLINK TOÀN CỤC /content/data
# -------------------------------------------------------------------------
!mkdir -p /content/data
!ln -sf /content/amodal_shape_prediction/data/train2014 /content/data/train2014
!ln -sf /content/amodal_shape_prediction/data/val2014 /content/data/val2014
!ln -sf /content/amodal_shape_prediction/data/annotations /content/data/annotations

print("\n" + "=" * 70)
print("🎉 THIẾT LẬP DỮ LIỆU COCOA HOÀN TẤT & SẴN SÀNG HUẤN LUYỆN!")
print(f"   • Train Annotations : {os.path.exists(os.path.join(ANN_DIR, 'COCO_amodal_train2014.json'))}")
print(f"   • Val Annotations   : {os.path.exists(os.path.join(ANN_DIR, 'COCO_amodal_val2014.json'))}")
train_cnt = len(os.listdir(train_img_dir)) if os.path.isdir(train_img_dir) else 0
val_cnt = len(os.listdir(val_img_dir)) if os.path.isdir(val_img_dir) else 0
print(f"   • Tổng ảnh Train    : {train_cnt:,} ảnh")
print(f"   • Tổng ảnh Val      : {val_cnt:,} ảnh")
print("=" * 70)

### 🚀 Bước 4: Huấn Luyện & Đánh Giá DUY NHẤT Row 7 (SG-GBA) Trên COCOA (~25 phút)
- **Kiến trúc:** Swin-UNet 5 kênh + Category Embedding (91 lớp) + SGGatedBottleneck (FiLM điều biến + Chú ý không gian 7x7 + Gate $\alpha$).
- **Số tham số chuẩn:** `EXPECTED_PARAMS = 34,867,566`.
- **Mục tiêu:** Chứng minh kiến trúc SG-GBA khắc phục triệt để hiện tượng tụt Invisible mIoU ở bài báo MAPR.

In [ ]:
%cd /content/amodal_shape_prediction/scripts/other_config_cocoa/amodal_shape_prediction_sg_gba_bottleneck

# 1. Huấn luyện 30 Epochs
!python train.py

# 2. Đánh giá kiểm thử khoa học
!python evaluate.py

---
# 🚗 PHẦN 2: TẬP DỮ LIỆU KINS (KITTI IN-HOUSE - 20 LỚP GIAO THÔNG)

### 🔍 Bước 5: Tự Động Quét Drive Hoặc Tải Dữ Liệu KINS (Smart Search & Fallback)
Cơ chế tự động:
1. **Kiểm tra trên Drive trước (`SEARCH_DIRS_KINS`):** Nếu có sẵn `kitti_data` và `kins_data` -> ưu tiên nối symlink dùng ngay.
2. **Nếu chưa có:** Tự động gọi `kagglehub` tải trực tiếp 2 dataset:
   - `anupammajhi/kitti-2d-object-detection` (Ảnh đường phố KITTI)
   - `ryanthenguyen/kins-data` (Annotations KINS đa giác amodal/visible)

In [ ]:
import os
import sys
import glob
import kagglehub

SEARCH_DIRS_KINS = [
    f"{DRIVE_DIR}/data/KINS",
    f"{DRIVE_DIR}/data/kins",
    f"{DRIVE_DIR}/data",
    "/content/drive/MyDrive/kins_data",
    "/content/drive/MyDrive/kitti_data",
    "/content/drive/MyDrive/data/KINS",
    "/content/drive/MyDrive/KINS",
    "/content/drive/MyDrive/CV+/data",
    "/content/drive/MyDrive/data",
    "/content/drive/MyDrive",
    "/content/amodal_shape_prediction/data/KINS",
    "/content"
]

def check_kins_ready():
    has_kitti = os.path.exists("/content/kitti_data/training/image_2") or os.path.exists("/content/kitti_data/data_object_image_2/training/image_2")
    has_kins = os.path.exists("/content/kins_data/update_train_2020.json") and os.path.exists("/content/kins_data/update_test_2020.json")
    return has_kitti and has_kins

print("=" * 70)
print("🔍 BẮT ĐẦU TỰ ĐỘNG THIẾT LẬP DỮ LIỆU KINS")
print("=" * 70)

if check_kins_ready():
    print("✅ Dữ liệu KINS và KITTI đã sẵn sàng trên Colab!")
else:
    # 1. Quét tìm KITTI
    kitti_drive = find_file_in_search_dirs(["kitti_data", "data_object_image_2", "*kitti*.zip"], SEARCH_DIRS_KINS)
    if kitti_drive and os.path.isdir(kitti_drive):
        print(f"🔗 Tìm thấy thư mục KITTI trên Drive: {kitti_drive}")
        !rm -rf /content/kitti_data && ln -sf "{kitti_drive}" /content/kitti_data
    else:
        print("🌐 Tự động tải KITTI 2D Object Detection từ Kaggle qua kagglehub...")
        kitti_path = kagglehub.dataset_download("anupammajhi/kitti-2d-object-detection")
        print(f"   ✅ Đã tải KITTI về: {kitti_path}")
        !rm -rf /content/kitti_data
        os.symlink(f"{kitti_path}/data_object_image_2", "/content/kitti_data")

    # 2. Quét tìm KINS Annotations
    kins_drive = find_file_in_search_dirs(["kins_data", "*kins_data*", "update_train_2020.json", "*kins*.zip"], SEARCH_DIRS_KINS)
    if kins_drive and os.path.isdir(kins_drive):
        print(f"🔗 Tìm thấy thư mục KINS trên Drive: {kins_drive}")
        !rm -rf /content/kins_data && ln -sf "{kins_drive}" /content/kins_data
    elif kins_drive and kins_drive.endswith("update_train_2020.json"):
        kins_folder = os.path.dirname(kins_drive)
        print(f"🔗 Tìm thấy file JSON KINS trên Drive: {kins_folder}")
        !rm -rf /content/kins_data && ln -sf "{kins_folder}" /content/kins_data
    else:
        print("🌐 Tự động tải KINS Data từ Kaggle (ryanthenguyen/kins-data)...")
        kins_path = kagglehub.dataset_download("ryanthenguyen/kins-data")
        print(f"   ✅ Đã tải KINS về: {kins_path}")
        !rm -rf /content/kins_data
        os.symlink(kins_path, "/content/kins_data")

# Chuẩn hóa nếu thư mục kitti bị lồng data_object_image_2
if not os.path.exists("/content/kitti_data/training/image_2") and os.path.exists("/content/kitti_data/data_object_image_2/training/image_2"):
    !ln -sf /content/kitti_data/data_object_image_2/* /content/kitti_data/

print("\n" + "=" * 70)
print("🎉 DỮ LIỆU KINS ĐÃ SẴN SÀNG HUẤN LUYỆN!")
print(f"   • KITTI Training Images : {os.path.exists('/content/kitti_data/training/image_2')}")
print(f"   • KINS Train JSON       : {os.path.exists('/content/kins_data/update_train_2020.json')}")
print(f"   • KINS Test JSON        : {os.path.exists('/content/kins_data/update_test_2020.json')}")
print("=" * 70)

### 🚀 Bước 6: Huấn Luyện & Đánh Giá DUY NHẤT Row 7 (SG-GBA) Trên KINS (~1.5 giờ)
- **Kiến trúc:** Swin-UNet 5 kênh + Category Embedding (20 lớp KINS) + SGGatedBottleneck.
- **Số tham số chuẩn:** `EXPECTED_PARAMS = 34,813,038`.
- **Batch Size:** 64 (tối ưu hóa cho VRAM của NVIDIA A100 và tăng tốc huấn luyện).

In [ ]:
%cd /content/amodal_shape_prediction

# 1. Huấn luyện 30 Epochs
!python scripts/train_kins_row7.py \
    --img-dir /content/kitti_data/training/image_2 \
    --ann-file /content/kins_data/update_train_2020.json \
    --save-dir checkpoints/kins/row7_sg_gba \
    --batch-size 64 \
    --epochs 30 \
    --num-workers 8 \
    --lr 1e-4

# 2. Đánh giá toàn diện trên tập Test
!python scripts/evaluate_kins_row7.py \
    --img-dir /content/kitti_data/testing/image_2 \
    --ann-file /content/kins_data/update_test_2020.json \
    --checkpoint checkpoints/kins/row7_sg_gba/swin_amodal_kins_row7_epoch_30.pth \
    --batch-size 64 \
    --output results/kins_row7_sg_gba_eval.json

---
# 📊 PHẦN 3: TỔNG HỢP KẾT QUẢ & BÀN GIAO CHO RYAN

Kiểm tra toàn bộ các file checkpoints `.pth` ở Epoch 30 và file kết quả `.json` đã được lưu bền vững trên Google Drive.

In [ ]:
import os
import json
import pandas as pd
from IPython.display import display

print("=" * 75)
print("🏆 BÁO CÁO TỔNG HỢP THỰC NGHIỆM ROW 7 (SG-GBA) TRÊN COLAB A100")
print("=" * 75)

summary_records = []

# 1. Đọc kết quả COCOA Row 7
cocoa_json = "/content/amodal_shape_prediction/results/cocoa_row7_sg_gba_eval.json"
if os.path.exists(cocoa_json):
    with open(cocoa_json, "r", encoding="utf-8") as f:
        d_cocoa = json.load(f)["summary"]
    summary_records.append({
        "Tập dữ liệu": "COCOA (91 classes)",
        "Cấu hình": "Row 7 — SG-GBA (Proposed)",
        "Overall mIoU": f"{d_cocoa.get('mIoU', 0.0):.2f}%",
        "Occlusion mIoU": f"{d_cocoa.get('Invisible_mIoU', 0.0):.2f}%",
        "Precision (mAP)": f"{d_cocoa.get('Precision', 0.0):.2f}%",
        "Recall (mAR)": f"{d_cocoa.get('Recall', 0.0):.2f}%",
        "Gate Alpha": d_cocoa.get("gate_alpha", "-"),
        "Trạng thái": "✅ Hoàn tất"
    })

# 2. Đọc kết quả KINS Row 7
kins_json = "/content/amodal_shape_prediction/results/kins_row7_sg_gba_eval.json"
if os.path.exists(kins_json):
    with open(kins_json, "r", encoding="utf-8") as f:
        d_kins = json.load(f)["summary"]
    summary_records.append({
        "Tập dữ liệu": "KINS (20 classes)",
        "Cấu hình": "Row 7 — SG-GBA (Proposed)",
        "Overall mIoU": f"{d_kins.get('mIoU', 0.0):.2f}%",
        "Occlusion mIoU": f"{d_kins.get('Occlusion_mIoU', 0.0):.2f}%",
        "Precision (mAP)": f"{d_kins.get('Precision', 0.0):.2f}%",
        "Recall (mAR)": f"{d_kins.get('Recall', 0.0):.2f}%",
        "Gate Alpha": d_kins.get("gate_alpha", "-"),
        "Trạng thái": "✅ Hoàn tất"
    })

if summary_records:
    df = pd.DataFrame(summary_records)
    display(df)
else:
    print("⚠️ Chưa tìm thấy file JSON kết quả trong results/")

print("\n📂 Danh sách Checkpoints đã lưu trên Google Drive:")
!ls -lh "{DRIVE_DIR}/checkpoints/cocoa/row7_sg_gba_bottleneck/" 2>/dev/null || true
!ls -lh "{DRIVE_DIR}/checkpoints/kins/row7_sg_gba/" 2>/dev/null || true
print("\n" + "=" * 75)
print("🎉 TOÀN BỘ FILE ĐÃ NẰM TRỌN TRONG GOOGLE DRIVE, SẴN SÀNG BÀN GIAO!")
print("=" * 75)